# Pairs Trading — Scenarios & Experimentation

This notebook is your sandbox. Use it to:

- **Run all configured pairs** through the full pipeline and compare them in one table
- **Tune parameters** (z-score thresholds, rolling window) and see the effect on signal behavior
- **Drill into any pair** with full charts
- **Compare parameter sets** side-by-side on the same pair

The key insight when experimenting: there's no single "correct" parameter set. Every choice involves a tradeoff between trade frequency, signal quality, and holding period. Your job as a quant is to understand *why* each parameter change has the effect it does — not just find the settings with the most signals or the fewest stops.

---

## How to use this notebook

1. **Start with Section 1** — run all pairs at once to get a high-level comparison
2. **Pick a pair** that interests you from the summary table
3. **Go to Section 2** — use the single config block to drill into that pair with full charts
4. **Go to Section 3** — try different parameter sets on the same pair and compare the outcomes
5. **Add your own pairs** in `pairs_config.py` and re-run

In [ ]:
import sys
import pandas as pd
sys.path.insert(0, '../..')

from src.strategies.pairs.config import PAIRS, DEFAULT_PARAMS
from src.data.fetcher import fetch_pair
from src.strategies.pairs.spread import compute_hedge_ratio, compute_spread
from src.analytics.stationarity import adf_test, compute_half_life
from src.strategies.pairs.signals import compute_zscore, generate_signals
from src.strategies.pairs.viz import plot_prices, plot_spread, plot_zscore

print('Configured pairs:')
for p in PAIRS:
    print(f"  {p['ticker_a']}/{p['ticker_b']}  ({p['sector']})")
print()
print('Default parameters:')
for k, v in DEFAULT_PARAMS.items():
    print(f'  {k:<20} = {v}')

---

## Section 1: Scan All Pairs

This cell runs every pair in `pairs_config.py` through the full pipeline using the default parameters and produces a single comparison table. It's the fastest way to get an overview of which pairs are statistically strong and which are weak.

**What each column means:**

| Column | What it tells you |
|---|---|
| `beta` | Hedge ratio — how many units of log(B) offset one unit of log(A) |
| `adf_pval` | ADF p-value — lower is better; < 0.05 means stationary |
| `stationary` | Whether the spread passes the stationarity test |
| `half_life_days` | Expected days to revert halfway to the mean |
| `long_entries` | Times we'd enter long the spread |
| `short_entries` | Times we'd enter short the spread |
| `exits` | Times the trade closed profitably (spread reverted) |
| `stops` | Times the trade hit the stop-loss |
| `stop_rate` | stops / (exits + stops) — fraction of closed trades that were stops |

A high `stop_rate` is a warning sign — the spread is moving persistently against the position more often than it reverts. A low `stop_rate` (< 20%) is healthy for a mean-reversion strategy.

In [2]:
params = DEFAULT_PARAMS
rows = []

for pair in PAIRS:
    ta, tb = pair['ticker_a'], pair['ticker_b']
    try:
        df = fetch_pair(ta, tb, period=params['period'])
        beta = compute_hedge_ratio(df['close_a'], df['close_b'])
        spread = compute_spread(df['close_a'], df['close_b'], beta)
        adf = adf_test(spread)
        hl = compute_half_life(spread)
        zscore = compute_zscore(spread, window=params['rolling_window'])
        sigs = generate_signals(zscore, params['z_entry'], params['z_exit'], params['z_stop'])
        counts = sigs['signal'].value_counts().to_dict()
        exits = counts.get('exit', 0)
        stops = counts.get('stop', 0)
        total_closed = exits + stops
        rows.append({
            'pair': f"{ta}/{tb}",
            'sector': pair['sector'],
            'beta': round(beta, 3),
            'adf_pval': adf['p_value'],
            'stationary': '✓' if adf['is_stationary'] else '✗',
            'half_life_days': hl,
            'long_entries': counts.get('long_spread', 0),
            'short_entries': counts.get('short_spread', 0),
            'exits': exits,
            'stops': stops,
            'stop_rate': f"{stops/total_closed:.0%}" if total_closed > 0 else 'n/a',
        })
    except Exception as e:
        rows.append({'pair': f"{ta}/{tb}", 'error': str(e)})

summary = pd.DataFrame(rows)
print(summary.to_string(index=False))

      pair           sector  beta  adf_pval stationary  half_life_days  long_entries  short_entries  exits  stops stop_rate
    KO/PEP Consumer Staples 0.001    0.7236          ✗           109.7             3              5      8      0        0%
   JPM/BAC       Financials 1.243    0.1366          ✗            30.2             4              5      9      0        0%
   XOM/CVX           Energy 1.094    0.2121          ✗            36.6             3              6      9      0        0%
MSFT/GOOGL       Technology 0.083    0.5146          ✗            65.4             4              3      7      0        0%


**How to read this table:**

- Sort mentally by `adf_pval` — the pairs closest to 0 have the strongest statistical case for pairs trading.
- Compare `half_life_days` across pairs. A pair with a 10-day half-life will trade much more actively than one with a 40-day half-life under the same parameters.
- Look at `stop_rate`. If more than ~30% of trades are hitting the stop, the z-score thresholds may be too tight for that pair's volatility, or the pair may not be as stable as the ADF test suggested.
- Notice that all pairs use the same `rolling_window = 60`. The half-life tells you whether that's a good fit — ideally the window is 2–3× the half-life. If a pair has a half-life of 10 days, a 60-day window may be too sluggish.

---

## Section 2: Deep Dive Into a Single Pair

**To use this section:** Change `TICKER_A` and `TICKER_B` to any pair you want to inspect. You can use the pairs from `pairs_config.py` or try any two tickers you're curious about.

The parameters block below is a local copy of `DEFAULT_PARAMS` — changes here only affect this section and don't modify the config file. This is intentional: it lets you experiment freely without breaking the rest of the notebook.

**Try these as starting points:**
- `'KO'` / `'PEP'` — classic consumer staples pair, usually very stable
- `'JPM'` / `'BAC'` — large-cap financials, more volatile
- `'XOM'` / `'CVX'` — energy majors, driven by oil prices
- `'MSFT'` / `'GOOGL'` — mega-cap tech; less fundamental similarity, riskier choice

In [3]:
# ── CHANGE THESE ──────────────────────────────────────────────────────────────
TICKER_A = 'KO'
TICKER_B = 'PEP'

PARAMS = {
    'period':         '2y',   # data lookback: '1y', '2y', '5y'
    'rolling_window': 60,     # days for z-score mean/std
    'z_entry':        2.0,    # enter when |z| exceeds this
    'z_exit':         0.5,    # exit when |z| falls below this
    'z_stop':         3.0,    # stop-loss when |z| exceeds this
}
# ── END CONFIG ────────────────────────────────────────────────────────────────

df_d = fetch_pair(TICKER_A, TICKER_B, period=PARAMS['period'])
beta_d = compute_hedge_ratio(df_d['close_a'], df_d['close_b'])
spread_d = compute_spread(df_d['close_a'], df_d['close_b'], beta_d)
adf_d = adf_test(spread_d)
hl_d = compute_half_life(spread_d)
zscore_d = compute_zscore(spread_d, window=PARAMS['rolling_window'])
signals_d = generate_signals(zscore_d, PARAMS['z_entry'], PARAMS['z_exit'], PARAMS['z_stop'])

counts_d = signals_d['signal'].value_counts().to_dict()
exits_d = counts_d.get('exit', 0)
stops_d = counts_d.get('stop', 0)
total_d = exits_d + stops_d

print(f'Pair: {TICKER_A} / {TICKER_B}   |   Period: {PARAMS["period"]}   |   {len(df_d)} trading days')
print()
print(f'  Hedge ratio (beta)  : {beta_d:.4f}')
print(f'  ADF p-value         : {adf_d["p_value"]}  {"✓ stationary" if adf_d["is_stationary"] else "✗ not stationary"}')
print(f'  Half-life           : {hl_d} days  (~{hl_d/5:.1f} weeks)')
print(f'  Suggested window    : {int(hl_d*2)}–{int(hl_d*3)} days  (using {PARAMS["rolling_window"]})')
print()
print(f'  Long entries        : {counts_d.get("long_spread", 0)}')
print(f'  Short entries       : {counts_d.get("short_spread", 0)}')
print(f'  Exits (profit)      : {exits_d}')
print(f'  Stops (loss)        : {stops_d}')
print(f'  Stop rate           : {stops_d/total_d:.0%}' if total_d > 0 else '  Stop rate           : n/a')

Pair: KO / PEP   |   Period: 2y   |   501 trading days

  Hedge ratio (beta)  : 0.0010
  ADF p-value         : 0.7236  ✗ not stationary
  Half-life           : 109.7 days  (~21.9 weeks)
  Suggested window    : 219–329 days  (using 60)

  Long entries        : 3
  Short entries       : 5
  Exits (profit)      : 8
  Stops (loss)        : 0
  Stop rate           : 0%


In [4]:
plot_prices(df_d, TICKER_A, TICKER_B).show()

In [5]:
plot_spread(spread_d, beta_d).show()

In [6]:
plot_zscore(signals_d, PARAMS['z_entry'], PARAMS['z_exit'], PARAMS['z_stop']).show()

---

## Section 3: Parameter Sensitivity

One of the most important skills in quant trading is understanding how sensitive your strategy is to its parameter choices. If changing `z_entry` from 2.0 to 1.8 doubles the number of trades, that suggests the strategy is highly sensitive near that threshold — which can be a sign of overfitting risk.

This section runs the same pair through **multiple parameter sets** and puts the results in a comparison table. 

### What to experiment with:

**`z_entry` (entry threshold):**
- Lower (e.g., 1.5): more trades, but more false signals — the spread hasn't deviated that much
- Higher (e.g., 2.5): fewer trades, but each one represents a more unusual deviation — stronger signal quality

**`rolling_window`:**
- Shorter (e.g., 20): the z-score reacts faster to recent moves, generates more signals, but is noisier
- Longer (e.g., 90): the z-score is smoother and more conservative, but slow to detect divergences
- Best practice: set it close to 2–3× the half-life of the pair

**`z_exit`:**
- Higher (e.g., 1.0): exit earlier, before full reversion — fewer bars in-position, but captures less of the move
- Lower (e.g., 0.25): wait for near-full reversion — more patient, but risks the spread reversing before you exit

**`z_stop`:**
- Tighter (e.g., 2.5): cut losses sooner — fewer large drawdowns but more stop-outs
- Wider (e.g., 4.0): give the trade more room — but a true regime change could cause a very large loss

### An important warning about this kind of analysis:

It's tempting to look at this table and pick the parameter set with the most exits and fewest stops. **Don't.** That's called **overfitting** — optimizing parameters to look good on historical data, which usually fails on new data. Think of it like studying the exact questions from last year's exam: you'll ace the test but won't actually understand the material.

Instead, use this table to understand the *direction* and *magnitude* of each parameter's effect, and choose parameters that make economic sense given the pair's half-life and volatility.

In [7]:
# ── CHANGE THESE ──────────────────────────────────────────────────────────────
# Pair to test (uses data already fetched in Section 2 if same ticker)
SENS_TICKER_A = 'KO'
SENS_TICKER_B = 'PEP'
SENS_PERIOD   = '2y'

# Define parameter sets to compare — add or remove rows freely
SCENARIOS = [
    {'label': 'Default',          'rolling_window': 60, 'z_entry': 2.0, 'z_exit': 0.5, 'z_stop': 3.0},
    {'label': 'Tighter entry',    'rolling_window': 60, 'z_entry': 1.5, 'z_exit': 0.5, 'z_stop': 3.0},
    {'label': 'Looser entry',     'rolling_window': 60, 'z_entry': 2.5, 'z_exit': 0.5, 'z_stop': 3.0},
    {'label': 'Short window',     'rolling_window': 20, 'z_entry': 2.0, 'z_exit': 0.5, 'z_stop': 3.0},
    {'label': 'Long window',      'rolling_window': 90, 'z_entry': 2.0, 'z_exit': 0.5, 'z_stop': 3.0},
    {'label': 'Early exit',       'rolling_window': 60, 'z_entry': 2.0, 'z_exit': 1.0, 'z_stop': 3.0},
    {'label': 'Tight stop',       'rolling_window': 60, 'z_entry': 2.0, 'z_exit': 0.5, 'z_stop': 2.5},
    {'label': 'Wide stop',        'rolling_window': 60, 'z_entry': 2.0, 'z_exit': 0.5, 'z_stop': 4.0},
]
# ── END CONFIG ────────────────────────────────────────────────────────────────

df_s = fetch_pair(SENS_TICKER_A, SENS_TICKER_B, period=SENS_PERIOD)
beta_s = compute_hedge_ratio(df_s['close_a'], df_s['close_b'])
spread_s = compute_spread(df_s['close_a'], df_s['close_b'], beta_s)

rows_s = []
for sc in SCENARIOS:
    z = compute_zscore(spread_s, window=sc['rolling_window'])
    sigs = generate_signals(z, sc['z_entry'], sc['z_exit'], sc['z_stop'])
    c = sigs['signal'].value_counts().to_dict()
    exits_s = c.get('exit', 0)
    stops_s = c.get('stop', 0)
    total_s = exits_s + stops_s
    pct_in_pos = (sigs['position'] != 0).mean()
    rows_s.append({
        'scenario':       sc['label'],
        'window':         sc['rolling_window'],
        'z_entry':        sc['z_entry'],
        'z_exit':         sc['z_exit'],
        'z_stop':         sc['z_stop'],
        'long_entries':   c.get('long_spread', 0),
        'short_entries':  c.get('short_spread', 0),
        'exits':          exits_s,
        'stops':          stops_s,
        'stop_rate':      f'{stops_s/total_s:.0%}' if total_s > 0 else 'n/a',
        'pct_days_in_pos': f'{pct_in_pos:.0%}',
    })

sens_df = pd.DataFrame(rows_s)
print(f'Sensitivity analysis: {SENS_TICKER_A}/{SENS_TICKER_B}  |  {SENS_PERIOD}\n')
print(sens_df.to_string(index=False))

Sensitivity analysis: KO/PEP  |  2y

     scenario  window  z_entry  z_exit  z_stop  long_entries  short_entries  exits  stops stop_rate pct_days_in_pos
      Default      60      2.0     0.5     3.0             3              5      8      0        0%             46%
Tighter entry      60      1.5     0.5     3.0             5              6     11      0        0%             53%
 Looser entry      60      2.5     0.5     3.0             2              2      4      0        0%             29%
 Short window      20      2.0     0.5     3.0             7              8     14      0        0%             33%
  Long window      90      2.0     0.5     3.0             3              3      6      0        0%             40%
   Early exit      60      2.0     1.0     3.0             3              5      8      0        0%             31%
   Tight stop      60      2.0     0.5     2.5             3              5      8      0        0%             46%
    Wide stop      60      2.0     

**Reading the sensitivity table:**

- `pct_days_in_pos`: what fraction of calendar days you'd have an open position. High values (> 50%) mean you're almost always in a trade — less selective, more exposed to regime changes.
- Watch for scenarios where `stops` is high but `exits` is low — the spread isn't reverting, it's trending. Those parameter sets are poorly calibrated for this pair.
- Compare `long_entries` vs `short_entries`. They should be roughly equal. A large imbalance suggests the spread has a directional drift — which means the pair may not be as cointegrated as the ADF test suggested, or the relationship has changed over the 2-year window.

**What questions to ask yourself:**
1. Which scenario has the most balanced `exits` vs `stops`?
2. How much does changing the window from 20 to 90 affect trade frequency?
3. Does tightening the entry threshold (1.5) generate proportionally more stops — suggesting those extra trades are noise?
4. Does a tight stop (2.5) lead to dramatically more stops than a wide one (4.0)?

---

## Section 4: Try Your Own Pair

This is the open-ended section. Pick any two US stocks you think might be cointegrated and run them through the full pipeline. 

**Good candidates to try:**
- **Retailers**: `WMT` / `TGT` (Walmart vs Target — both big-box US retailers)
- **Airlines**: `DAL` / `UAL` (Delta vs United — same routes, same fuel costs)
- **Semiconductors**: `NVDA` / `AMD` (both GPU makers, but very different valuations — might surprise you)
- **Pharma**: `JNJ` / `ABT` (both large diversified healthcare companies)
- **Banks**: `GS` / `MS` (Goldman Sachs vs Morgan Stanley — investment banking peers)

**What to look for:**
1. Does the spread chart look stationary (oscillating around a mean) or does it trend?
2. Is the ADF p-value below 0.05?
3. What's the half-life — does it match the kind of trade you'd want to hold?
4. Are there more exits than stops, or more stops than exits?

**What often surprises people:**
- Two stocks in the same sector aren't always cointegrated. Company-specific events (product launches, CEO changes, lawsuits) can permanently shift the relationship.
- NVDA/AMD is a classic example: both make GPUs, but NVDA's AI-driven growth created a structural divergence starting around 2022. The ADF test will likely fail, or the spread will show a clear directional trend.
- Some pairs that *look* similar (like two airlines) can fail the test during certain periods because of idiosyncratic events (one airline had a major incident, the other got a favorable merger ruling, etc.).

In [8]:
# ── CHANGE THESE ──────────────────────────────────────────────────────────────
MY_TICKER_A = 'WMT'
MY_TICKER_B = 'TGT'
MY_PERIOD   = '2y'
MY_WINDOW   = 60
# ── END CONFIG ────────────────────────────────────────────────────────────────

df_m = fetch_pair(MY_TICKER_A, MY_TICKER_B, period=MY_PERIOD)
beta_m = compute_hedge_ratio(df_m['close_a'], df_m['close_b'])
spread_m = compute_spread(df_m['close_a'], df_m['close_b'], beta_m)
adf_m = adf_test(spread_m)
hl_m = compute_half_life(spread_m)
zscore_m = compute_zscore(spread_m, window=MY_WINDOW)
signals_m = generate_signals(zscore_m, DEFAULT_PARAMS['z_entry'], DEFAULT_PARAMS['z_exit'], DEFAULT_PARAMS['z_stop'])

counts_m = signals_m['signal'].value_counts().to_dict()
exits_m = counts_m.get('exit', 0)
stops_m = counts_m.get('stop', 0)
total_m = exits_m + stops_m

print(f'Pair: {MY_TICKER_A} / {MY_TICKER_B}   |   Period: {MY_PERIOD}   |   {len(df_m)} trading days')
print()
print(f'  Hedge ratio (beta)  : {beta_m:.4f}')
print(f'  ADF p-value         : {adf_m["p_value"]}  {"✓ stationary" if adf_m["is_stationary"] else "✗ NOT stationary — this pair may not be suitable"}')
print(f'  Half-life           : {hl_m} days  (~{hl_m/5:.1f} weeks)')
print()
print(f'  Long entries        : {counts_m.get("long_spread", 0)}')
print(f'  Short entries       : {counts_m.get("short_spread", 0)}')
print(f'  Exits               : {exits_m}')
print(f'  Stops               : {stops_m}')
if total_m > 0:
    print(f'  Stop rate           : {stops_m/total_m:.0%}')

Pair: WMT / TGT   |   Period: 2y   |   501 trading days

  Hedge ratio (beta)  : -0.5362
  ADF p-value         : 0.6777  ✗ NOT stationary — this pair may not be suitable
  Half-life           : 101.8 days  (~20.4 weeks)

  Long entries        : 1
  Short entries       : 7
  Exits               : 8
  Stops               : 0
  Stop rate           : 0%


In [9]:
plot_prices(df_m, MY_TICKER_A, MY_TICKER_B).show()

In [10]:
plot_spread(spread_m, beta_m).show()

In [11]:
plot_zscore(signals_m, DEFAULT_PARAMS['z_entry'], DEFAULT_PARAMS['z_exit'], DEFAULT_PARAMS['z_stop']).show()

---

## Quick Reference: Diagnosing Problems

Use this table when something looks off in your results:

| Symptom | Likely cause | What to try |
|---|---|---|
| ADF p-value > 0.05 | Spread isn't stationary — pair may not be cointegrated | Try a different pair, or a shorter `period` (more recent history) |
| Half-life > 60 days | Very slow mean reversion | Use a longer `rolling_window`; widen `z_entry` to 2.5+ |
| Half-life < 5 days | Very fast mean reversion | Daily bars may be too slow; consider shorter-term data |
| More stops than exits | Spread trends persistently after entry | Widen `z_entry` to 2.5+, tighten `z_stop`, or reconsider the pair |
| Very few signals (0–2) | Thresholds too wide for this pair's volatility | Lower `z_entry` to 1.5–1.8 |
| Signals all clustered in one period | Pair had a one-time divergence event | Try a longer `period` to get more history |
| Long and short entries very unbalanced | Spread has directional drift | Pair may be co-trending, not cointegrating; check the spread chart for a clear trend |
| Window suggestion >> default 60 | Half-life is long | Increase `rolling_window` to match; signals will be fewer but better calibrated |

---

## What's Not Here Yet

This notebook shows you *which trades would have been signaled* but not *how much money they would have made*. That's the job of the backtest engine — the next thing to build. The backtest will take the `signals` DataFrame and simulate actual P&L, including:

- Dollar P&L on each trade (entry price → exit/stop price)
- Equity curve over time
- Sharpe ratio (return per unit of risk)
- Maximum drawdown (worst peak-to-trough loss)
- Win rate and average win/loss size

Once you have a backtest, the sensitivity analysis in Section 3 becomes much more powerful — instead of comparing signal counts, you'll compare actual risk-adjusted returns.